# BERT Sentence Relation Classifier
Predicts the relation between two sentences: **opposite**, **same**, or **limited**.

**Pipeline:**
1. Load MultiNLI dataset
2. Remap NLI labels → custom relation labels
3. Fine-tune `bert-base-uncased`
4. Evaluate with accuracy + F1
5. Save & run inference

## 1. Install dependencies

In [ ]:
!pip install transformers datasets evaluate scikit-learn accelerate -q

## 2. Imports

In [ ]:
import torch
import numpy as np
import evaluate
from datasets import load_dataset
from transformers import (
    BertTokenizer,
    BertForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
)

print(f"PyTorch version : {torch.__version__}")
print(f"GPU available   : {torch.cuda.is_available()}")
print(f"Device          : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

## 3. Config

In [ ]:
# ── Model ──────────────────────────────────────────────────────────────────────
MODEL_NAME = "bert-base-uncased"
# Alternatively use a stronger NLI-pretrained checkpoint (recommended):
# MODEL_NAME = "cross-encoder/nli-MiniLM2-L6-H768"   # fast & accurate
# MODEL_NAME = "roberta-large-mnli"                   # highest accuracy
# MODEL_NAME = "typeform/distilbert-base-uncased-mnli" # lightweight

MAX_LEN        = 128
BATCH_TRAIN    = 16
BATCH_EVAL     = 32
EPOCHS         = 5
LR             = 2e-5
WEIGHT_DECAY   = 0.01
WARMUP_RATIO   = 0.1
OUTPUT_DIR     = "./bert-relation"
FINAL_DIR      = "./bert-relation-final"

# ── Labels ─────────────────────────────────────────────────────────────────────
# Your custom relation labels
LABELS   = ["opposite", "same", "limited"]   # extend as needed
label2id = {l: i for i, l in enumerate(LABELS)}
id2label = {i: l for l, i in label2id.items()}

# MultiNLI gold labels → your custom labels
# MultiNLI uses:  0=entailment  1=neutral  2=contradiction  (-1=unlabeled)
MNLI_TO_CUSTOM = {
    0: label2id["same"],       # entailment   → same
    1: label2id["limited"],    # neutral      → limited
    2: label2id["opposite"],   # contradiction→ opposite
}

print("Label mapping:", MNLI_TO_CUSTOM)
print("id2label     :", id2label)

## 4. Load & remap MultiNLI dataset

In [ ]:
raw = load_dataset("multi_nli")

# MultiNLI splits: train / validation_matched / validation_mismatched
print(raw)
print("\nSample row:")
print(raw["train"][0])

In [ ]:
def remap_labels(example):
    """
    MultiNLI label integers → custom relation integers.
    Drops unlabeled rows (label == -1).
    """
    example["label"] = MNLI_TO_CUSTOM.get(example["label"], -1)
    return example

# Apply remapping and drop unlabeled rows
dataset = (
    raw
    .map(remap_labels)
    .filter(lambda x: x["label"] != -1)
)

train_ds  = dataset["train"]
val_ds    = dataset["validation_matched"]   # in-domain validation

print(f"Train size : {len(train_ds):,}")
print(f"Val size   : {len(val_ds):,}")

## 5. Tokenize

In [ ]:
tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)

def preprocess(examples):
    """
    Tokenizes sentence pairs.
    Input format fed to BERT: [CLS] sentence1 [SEP] sentence2 [SEP]
    (equivalent to your [REL] sent1 [SEP] sent2 format — [CLS] acts as [REL])
    """
    enc = tokenizer(
        examples["premise"],          # sentence 1  (MultiNLI column name)
        examples["hypothesis"],       # sentence 2
        truncation=True,
        padding="max_length",
        max_length=MAX_LEN,
    )
    enc["labels"] = examples["label"]
    return enc

REMOVE_COLS = [c for c in train_ds.column_names if c not in ["label"]]

train_ds = train_ds.map(preprocess, batched=True, remove_columns=REMOVE_COLS)
val_ds   = val_ds.map(preprocess,   batched=True, remove_columns=REMOVE_COLS)

train_ds.set_format("torch")
val_ds.set_format("torch")

print("Tokenized columns:", train_ds.column_names)
print("Sample input_ids :", train_ds[0]["input_ids"][:12].tolist(), "...")

## 6. Load pretrained model

> **Tip:** swap `MODEL_NAME` in the Config cell to `roberta-large-mnli` for a model that is already NLI-pretrained — it will converge faster and score higher.

In [ ]:
model = BertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABELS),
    id2label=id2label,
    label2id=label2id,
    hidden_dropout_prob=0.1,
    attention_probs_dropout_prob=0.1,
)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params    : {total_params:,}")
print(f"Trainable params: {trainable_params:,}")

## 7. Metrics

In [ ]:
accuracy_metric = evaluate.load("accuracy")
f1_metric       = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_metric.compute(predictions=preds, references=labels)["accuracy"]
    f1  = f1_metric.compute(
        predictions=preds,
        references=labels,
        average="macro",          # equal weight per class
    )["f1"]
    return {"accuracy": acc, "f1_macro": f1}

## 8. Training

In [ ]:
training_args = TrainingArguments(
    output_dir                  = OUTPUT_DIR,
    num_train_epochs            = EPOCHS,
    per_device_train_batch_size = BATCH_TRAIN,
    per_device_eval_batch_size  = BATCH_EVAL,
    learning_rate               = LR,
    weight_decay                = WEIGHT_DECAY,
    warmup_ratio                = WARMUP_RATIO,
    evaluation_strategy         = "epoch",
    save_strategy               = "epoch",
    load_best_model_at_end      = True,
    metric_for_best_model       = "f1_macro",   # save best by F1, not loss
    greater_is_better           = True,
    logging_steps               = 200,
    report_to                   = "none",        # set "wandb" if you use W&B
    fp16                        = torch.cuda.is_available(),  # auto mixed precision
    dataloader_num_workers      = 2,
)

trainer = Trainer(
    model           = model,
    args            = training_args,
    train_dataset   = train_ds,
    eval_dataset    = val_ds,
    compute_metrics = compute_metrics,
    callbacks       = [EarlyStoppingCallback(early_stopping_patience=2)],
)

trainer.train()

## 9. Evaluate

In [ ]:
results = trainer.evaluate()
print("\n── Evaluation results ──")
for k, v in results.items():
    print(f"  {k:35s}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

In [ ]:
# Per-class breakdown with sklearn classification report
from sklearn.metrics import classification_report

preds_output = trainer.predict(val_ds)
preds        = np.argmax(preds_output.predictions, axis=-1)
true_labels  = preds_output.label_ids

print(classification_report(
    true_labels,
    preds,
    target_names=[id2label[i] for i in range(len(LABELS))],
))

## 10. Save model

In [ ]:
trainer.save_model(FINAL_DIR)
tokenizer.save_pretrained(FINAL_DIR)
print(f"Model saved to {FINAL_DIR}")

## 11. Inference

Load the saved model and predict the relation for any two sentences.

In [ ]:
from transformers import pipeline

# ── Load from saved checkpoint ──────────────────────────────────────────────
classifier = pipeline(
    "text-classification",
    model=FINAL_DIR,
    tokenizer=FINAL_DIR,
    device=0 if torch.cuda.is_available() else -1,
)

def predict_relation(sentence1: str, sentence2: str) -> dict:
    """
    Returns predicted relation label and confidence score.
    Input format: [CLS] sentence1 [SEP] sentence2 [SEP]
    """
    result = classifier(
        f"{sentence1} [SEP] {sentence2}",
        truncation=True,
        max_length=MAX_LEN,
    )[0]
    return {"relation": result["label"], "confidence": round(result["score"], 4)}


# ── Test examples ────────────────────────────────────────────────────────────
examples = [
    ("The sun rises in the east.",       "The sun sets in the west."),
    ("Dogs are mammals.",                "Canines belong to the mammal family."),
    ("Water is wet.",                    "Fire is hot."),
    ("The economy grew last quarter.",   "GDP declined sharply this year."),
]

print(f"{'Sentence 1':<40} {'Sentence 2':<40} {'Relation':<12} {'Confidence'}")
print("-" * 105)
for s1, s2 in examples:
    out = predict_relation(s1, s2)
    print(f"{s1:<40} {s2:<40} {out['relation']:<12} {out['confidence']:.4f}")

## 12. (Optional) Fine-tune further on your own data

If you have domain-specific sentence pairs, add them here to boost performance on your custom distribution.

In [ ]:
from datasets import Dataset

# Replace with your own labeled pairs
my_data = {
    "sentence1": [
        "The product is expensive.",
        "It rained heavily today.",
    ],
    "sentence2": [
        "The item costs very little.",
        "There was a severe downpour this morning.",
    ],
    "label": ["opposite", "same"],
}

def preprocess_custom(examples):
    enc = tokenizer(
        examples["sentence1"],
        examples["sentence2"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LEN,
    )
    enc["labels"] = [label2id[l] for l in examples["label"]]
    return enc

custom_ds = Dataset.from_dict(my_data).map(preprocess_custom, batched=True)
custom_ds.set_format("torch")

# Fine-tune the already-trained model for 2 more epochs on custom data
fine_tune_args = TrainingArguments(
    output_dir                  = "./bert-relation-custom",
    num_train_epochs            = 2,
    per_device_train_batch_size = 8,
    learning_rate               = 1e-5,      # lower LR to avoid forgetting
    weight_decay                = 0.01,
    evaluation_strategy         = "no",
    save_strategy               = "epoch",
    fp16                        = torch.cuda.is_available(),
    report_to                   = "none",
)

fine_tuner = Trainer(
    model         = model,
    args          = fine_tune_args,
    train_dataset = custom_ds,
)
fine_tuner.train()

model.save_pretrained("./bert-relation-custom-final")
tokenizer.save_pretrained("./bert-relation-custom-final")
print("Custom fine-tuned model saved.")